# Sentinel - Data Exploration

Quick look at what's in the database: label distribution, ticker breakdown, exaggeration scores, and temporal patterns.

In [ ]:
import sys
sys.path.insert(0, "..")

import os
from collections import Counter
from dotenv import load_dotenv
import psycopg
import matplotlib.pyplot as plt

load_dotenv("../.env")

conn = psycopg.connect(os.environ["DATABASE_URL"])
print("Connected")

## Overview

In [ ]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM raw_claims")
    raw_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM labeled_claims")
    labeled_count = cur.fetchone()[0]

print(f"Raw claims:     {raw_count:,}")
print(f"Labeled claims: {labeled_count:,}")
print(f"Unlabeled:      {raw_count - labeled_count:,}")

## Label distribution

In [ ]:
with conn.cursor() as cur:
    cur.execute("SELECT label, COUNT(*) FROM labeled_claims GROUP BY label ORDER BY COUNT(*) DESC")
    label_dist = dict(cur.fetchall())

total = sum(label_dist.values())
for label, count in label_dist.items():
    print(f"{label:>15}: {count:>5}  ({count/total:.1%})")

fig, ax = plt.subplots(figsize=(6, 4))
colors = {"accurate": "#4CAF50", "exaggerated": "#F44336", "understated": "#2196F3"}
bars = ax.bar(label_dist.keys(), label_dist.values(),
              color=[colors.get(l, "#999") for l in label_dist.keys()])
ax.bar_label(bars, fmt="%d")
ax.set_ylabel("Count")
ax.set_title("Label distribution")
plt.tight_layout()
plt.show()

## Claims by ticker

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT r.ticker, l.label, COUNT(*)
        FROM raw_claims r
        JOIN labeled_claims l ON r.tweet_id = l.tweet_id
        GROUP BY r.ticker, l.label
        ORDER BY r.ticker, l.label
    """)
    rows = cur.fetchall()

# Pivot into {ticker: {label: count}}
tickers = {}
for ticker, label, count in rows:
    tickers.setdefault(ticker, {})[label] = count

# Sort by total count descending
sorted_tickers = sorted(tickers.items(), key=lambda x: sum(x[1].values()), reverse=True)

fig, ax = plt.subplots(figsize=(10, 5))
labels_order = ["accurate", "exaggerated", "understated"]
bottom = [0] * len(sorted_tickers)

for label in labels_order:
    vals = [t[1].get(label, 0) for t in sorted_tickers]
    ax.bar([t[0] for t in sorted_tickers], vals, bottom=bottom,
           label=label, color=colors.get(label, "#999"))
    bottom = [b + v for b, v in zip(bottom, vals)]

ax.set_ylabel("Count")
ax.set_title("Claims by ticker")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Exaggeration score distribution

In [ ]:
with conn.cursor() as cur:
    cur.execute("SELECT exaggeration_score, label FROM labeled_claims")
    scores = cur.fetchall()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall histogram
all_scores = [s[0] for s in scores]
axes[0].hist(all_scores, bins=30, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Exaggeration score")
axes[0].set_ylabel("Count")
axes[0].set_title("Exaggeration score distribution (all)")

# By label
for label in ["accurate", "exaggerated", "understated"]:
    label_scores = [s[0] for s in scores if s[1] == label]
    if label_scores:
        axes[1].hist(label_scores, bins=20, alpha=0.5,
                     label=f"{label} (n={len(label_scores)})",
                     color=colors.get(label, "#999"))

axes[1].set_xlabel("Exaggeration score")
axes[1].set_ylabel("Count")
axes[1].set_title("Exaggeration score by label")
axes[1].legend()

plt.tight_layout()
plt.show()

## Claims over time

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT DATE(r.created_at) as day, l.label, COUNT(*)
        FROM raw_claims r
        JOIN labeled_claims l ON r.tweet_id = l.tweet_id
        GROUP BY day, l.label
        ORDER BY day
    """)
    rows = cur.fetchall()

from collections import defaultdict
daily = defaultdict(lambda: defaultdict(int))
for day, label, count in rows:
    daily[day][label] = count

days = sorted(daily.keys())

fig, ax = plt.subplots(figsize=(12, 4))
for label in ["accurate", "exaggerated", "understated"]:
    vals = [daily[d].get(label, 0) for d in days]
    ax.plot(days, vals, label=label, color=colors.get(label, "#999"), marker=".", markersize=3)

ax.set_ylabel("Claims per day")
ax.set_title("Claims over time")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Price change distribution by label

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT r.price_change_pct, l.label
        FROM raw_claims r
        JOIN labeled_claims l ON r.tweet_id = l.tweet_id
        WHERE r.price_change_pct IS NOT NULL
    """)
    rows = cur.fetchall()

fig, ax = plt.subplots(figsize=(8, 4))
for label in ["accurate", "exaggerated"]:
    pcts = [r[0] for r in rows if r[1] == label]
    if pcts:
        ax.hist(pcts, bins=40, alpha=0.5, label=f"{label} (n={len(pcts)})",
                color=colors.get(label, "#999"))

ax.set_xlabel("24h price change (%)")
ax.set_ylabel("Count")
ax.set_title("Price change distribution by label")
ax.legend()
plt.tight_layout()
plt.show()

## Top 10 most exaggerated users

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT r.username, COUNT(*) as cnt,
               SUM(CASE WHEN l.label = 'exaggerated' THEN 1 ELSE 0 END) as exaggerated,
               ROUND(AVG(l.exaggeration_score)::numeric, 3) as avg_score
        FROM raw_claims r
        JOIN labeled_claims l ON r.tweet_id = l.tweet_id
        GROUP BY r.username
        HAVING COUNT(*) >= 5
        ORDER BY SUM(CASE WHEN l.label = 'exaggerated' THEN 1 ELSE 0 END)::float / COUNT(*) DESC
        LIMIT 15
    """)
    columns = [desc[0] for desc in cur.description]
    rows = cur.fetchall()

print(f"{'Username':>20} {'Total':>6} {'Exagg':>6} {'Rate':>8} {'Avg Score':>10}")
print("-" * 55)
for row in rows:
    username, total, exagg, avg_score = row
    rate = exagg / total
    print(f"{username:>20} {total:>6} {exagg:>6} {rate:>7.1%} {float(avg_score):>10.3f}")

## Catalyst breakdown

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT r.catalyst_type, l.label, COUNT(*)
        FROM raw_claims r
        JOIN labeled_claims l ON r.tweet_id = l.tweet_id
        GROUP BY r.catalyst_type, l.label
        ORDER BY r.catalyst_type, l.label
    """)
    rows = cur.fetchall()

catalysts = {}
for cat_type, label, count in rows:
    cat_name = cat_type or "none"
    catalysts.setdefault(cat_name, {})[label] = count

print(f"{'Catalyst':>15} {'Accurate':>10} {'Exaggerated':>12} {'Understated':>12} {'Total':>8} {'Exagg Rate':>12}")
print("-" * 72)
for cat, dist in sorted(catalysts.items(), key=lambda x: sum(x[1].values()), reverse=True):
    acc = dist.get("accurate", 0)
    exg = dist.get("exaggerated", 0)
    und = dist.get("understated", 0)
    total = acc + exg + und
    rate = exg / total if total else 0
    print(f"{cat:>15} {acc:>10} {exg:>12} {und:>12} {total:>8} {rate:>11.1%}")

## Text length distribution

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT LENGTH(r.text), l.label
        FROM raw_claims r
        JOIN labeled_claims l ON r.tweet_id = l.tweet_id
    """)
    rows = cur.fetchall()

fig, ax = plt.subplots(figsize=(8, 4))
for label in ["accurate", "exaggerated"]:
    lengths = [r[0] for r in rows if r[1] == label]
    if lengths:
        ax.hist(lengths, bins=30, alpha=0.5, label=f"{label} (n={len(lengths)})",
                color=colors.get(label, "#999"))

ax.set_xlabel("Tweet text length (chars)")
ax.set_ylabel("Count")
ax.set_title("Text length by label")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
conn.close()
print("Done")